# Day 4: Training the Memory Module
**Runtime:** A100 GPU. Select Runtime > Change runtime type > A100.

**Strategy:** Freeze TRIBE v2 (177M), train only MemoryAttention (5.3M).

In [ ]:
!pip install numpy==2.2.6 -q
# After this: Runtime > Restart runtime, then continue from next cell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/Research/memory-brain-encoding'
CACHE_DIR = os.path.join(PROJECT_DIR, 'cache')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CHECKPOINTS_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
for d in [CACHE_DIR, RESULTS_DIR, DATA_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)
print('Drive mounted!')

In [ ]:
%%capture
!pip install git+https://github.com/facebookresearch/tribev2.git
!pip install nilearn

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
except: pass
import torch, numpy as np
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1. Load Model + Memory Module

In [ ]:
from tribev2 import TribeModel
print('Loading TRIBE v2...')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE_DIR)
brain_model = model._model
print(f'Loaded on {brain_model.device}')

In [ ]:
!git clone https://github.com/Mrabbi3/memory-augmented-brain-encoding.git /content/my_repo 2>/dev/null || true
!cd /content/my_repo && git pull
import sys; sys.path.insert(0, '/content/my_repo/src')
from memory import MemoryBuffer, MemoryAttention, MemoryAugmentedEncoder
print('Memory module imported!')

In [ ]:
mem_encoder = MemoryAugmentedEncoder(brain_model=brain_model, buffer_size=100, top_k=5, num_heads=8)
for param in brain_model.parameters(): param.requires_grad = False
trainable = sum(p.numel() for p in mem_encoder.memory_attention.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in brain_model.parameters())
print(f'Frozen: {frozen:,} | Trainable: {trainable:,} ({trainable/frozen*100:.2f}%)')

---
## 2. Generate Training Video

In [ ]:
video_path = os.path.join(DATA_DIR, 'training_video_5min.mp4')
if not os.path.exists(video_path):
    os.system(f'ffmpeg -f lavfi -i "testsrc=duration=300:size=320x240:rate=24" -f lavfi -i "sine=frequency=440:duration=300" -shortest -y {video_path} 2>/dev/null')
    print('Created 5-min video')
else: print('Video exists')

In [ ]:
print('Extracting features (10-15 min on A100)...')
df = model.get_events_dataframe(video_path=video_path)
print(f'Extracted {len(df)} events')

In [ ]:
loader = model.data.get_loaders(events=df, split_to_build='all')['all']
print(f'DataLoader: {len(loader)} batches')

---
## 3. Training Loop

In [ ]:
import torch.optim as optim
from torch.nn import functional as F
LR = 1e-4; EPOCHS = 5
optimizer = optim.AdamW(mem_encoder.memory_attention.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print(f'Ready: AdamW lr={LR}, {EPOCHS} epochs')

In [ ]:
print('='*60)
print('TRAINING THE MEMORY MODULE')
print('='*60)
training_log = []
mem_encoder.memory_attention.train()
for epoch in range(EPOCHS):
    epoch_losses = []
    mem_encoder.reset_memory()
    prev_pred = None
    for batch_idx, batch in enumerate(loader):
        batch = batch.to(brain_model.device)
        optimizer.zero_grad()
        with torch.no_grad():
            x = brain_model.aggregate_features(batch)
            sid = batch.data.get('subject_id', None)
            if hasattr(brain_model, 'temporal_smoothing'):
                x = brain_model.temporal_smoothing(x.transpose(1,2)).transpose(1,2)
            if not brain_model.config.linear_baseline:
                x = brain_model.transformer_forward(x, sid)
        query = x.mean(dim=1)
        retrieved = mem_encoder.memory_buffer.retrieve(query)
        x_mem = mem_encoder.memory_attention(x, retrieved)
        mem_encoder.memory_buffer.store(x_mem)
        with torch.no_grad():
            xo = x_mem.transpose(1,2)
            if brain_model.config.low_rank_head is not None:
                xo = brain_model.low_rank_head(xo.transpose(1,2)).transpose(1,2)
            pred = brain_model.predictor(xo, sid)
            pred = brain_model.pooler(pred)
        if pred.shape[2] > 1:
            td = pred[:,:,1:] - pred[:,:,:-1]
            smooth_loss = td.pow(2).mean()
        else: smooth_loss = torch.tensor(0.0).to(pred.device)
        if prev_pred is not None:
            consist_loss = F.mse_loss(pred[:,:,0], prev_pred[:,:,-1].detach())
        else: consist_loss = torch.tensor(0.0).to(pred.device)
        gate_val = torch.tanh(mem_encoder.memory_attention.gate)
        gate_loss = -0.01 * gate_val.abs()
        loss = smooth_loss + 0.5 * consist_loss + gate_loss
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
        prev_pred = pred.detach()
    scheduler.step()
    avg = np.mean(epoch_losses)
    gv = torch.tanh(mem_encoder.memory_attention.gate).item()
    training_log.append({'epoch': epoch+1, 'loss': avg, 'gate': gv})
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg:.6f} | Gate: {gv:.4f}')
print('\nTraining complete!')

---
## 4. Training Curves

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = [x['epoch'] for x in training_log]
ax1.plot(epochs, [x['loss'] for x in training_log], 'b-o')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Loss')
ax2.plot(epochs, [x['gate'] for x in training_log], 'r-o')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Gate'); ax2.set_title('Memory Gate Value')
ax2.set_ylim(-0.1, 1.0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Compare Vanilla vs Trained Memory

In [ ]:
mem_encoder.memory_attention.eval()
mem_encoder.reset_memory()
preds_trained = []
with torch.inference_mode():
    for batch in loader:
        batch = batch.to(brain_model.device)
        y = mem_encoder.forward_with_memory(batch)
        preds_trained.append(y.detach().cpu().numpy())
from einops import rearrange
preds_trained = np.concatenate(preds_trained)
if preds_trained.ndim == 3: preds_trained = rearrange(preds_trained, 'b d t -> (b t) d')
preds_vanilla, _ = model.predict(events=df, verbose=False)
min_t = min(preds_vanilla.shape[0], preds_trained.shape[0])
diff = np.abs(preds_trained[:min_t] - preds_vanilla[:min_t]).mean(axis=0)
print(f'Mean diff after training: {diff.mean():.6f}')
print(f'Gate: {torch.tanh(mem_encoder.memory_attention.gate).item():.4f}')

In [ ]:
from nilearn import plotting, datasets
fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
nv = diff.shape[0] // 2
fig, axes = plt.subplots(2, 2, figsize=(14, 10), subplot_kw={'projection': '3d'})
for i, (hemi, view) in enumerate([('left','lateral'),('right','lateral'),('left','medial'),('right','medial')]):
    ax = axes[i//2][i%2]
    d = diff[:nv] if hemi=='left' else diff[nv:]
    mesh_key = f'pial_{hemi}'
    plotting.plot_surf_stat_map(fsaverage[mesh_key], d, hemi=hemi, view=view, colorbar=True, cmap='Reds', title=f'{hemi.title()} {view}', axes=ax, figure=fig)
plt.suptitle('Memory Impact After Training', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'memory_impact_trained.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Save Checkpoint

In [ ]:
ckpt = {'state_dict': mem_encoder.memory_attention.state_dict(), 'gate': torch.tanh(mem_encoder.memory_attention.gate).item(), 'log': training_log, 'config': {'buffer_size':100,'top_k':5,'num_heads':8,'hidden_dim':1152,'lr':LR,'epochs':EPOCHS}}
ckpt_path = os.path.join(CHECKPOINTS_DIR, 'memory_attention_v1.pt')
torch.save(ckpt, ckpt_path)
print(f'Saved: {ckpt_path} ({os.path.getsize(ckpt_path)/1e6:.1f} MB)')

In [ ]:
import json
from datetime import datetime
session = {'date': datetime.now().isoformat(), 'notebook': '04_training', 'epochs': EPOCHS, 'final_loss': float(training_log[-1]['loss']), 'final_gate': float(training_log[-1]['gate']), 'trainable_params': trainable, 'mean_diff': float(diff.mean()), 'status': 'completed'}
log_path = os.path.join(RESULTS_DIR, 'session_log.json')
logs = json.load(open(log_path)) if os.path.exists(log_path) else []
logs.append(session)
json.dump(logs, open(log_path,'w'), indent=2)
print('Day 4 Complete!')
print(f'Gate: {training_log[-1]["gate"]:.4f}')
print(f'Loss: {training_log[-1]["loss"]:.6f}')

---
## What's Next

You trained the memory module! The gate should now be > 0, meaning memory is contributing.

**For the full paper**, you need to:
1. Get CNeuroMod fMRI data (request access at cneuromod.ca)
2. Train with MSE loss against real fMRI recordings
3. Compute per-region encoding scores (Pearson R)
4. Show hippocampus/prefrontal improvement
5. Run ablation experiments